# Transformer Based Classification

This notebook implements a RoBERTa-based model for emotion classification using the Hugging Face transformers library.

## 0. Setup and Dependencies

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset
import evaluate
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
project_root = Path().absolute().parent
sys.path.append(str(project_root))

from emotion_classifier.data import load_raw_data, prepare_dataset_splits

# Load dataset
train_df, test_df = load_raw_data()
print(f"Loaded dataset with {len(train_df)} samples")

# Create train/validation/test splits if not already done
try:
    train_df, val_df, test_df = prepare_dataset_splits(train_df)
    print(f"Created splits: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")
except Exception as e:
    print(f"Using existing splits: {e}")
    
# Preview the data
display(train_df.head())

# Define label mapping for consistent evaluation
label_mapping = {"Mixed": 0, "Positive": 1, "Negative": 2, "Ambiguous": 3, "Neutral": 4}
id2label = {v: k for k, v in label_mapping.items()}
label2id = label_mapping

print("\nAvailable labels:", list(label_mapping.keys()))

/home/francisco/Desktop/MEIC_1YEAR/2semestre/PLN/pln/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded dataset with 1874 samples
Created splits: train=1124, val=375, test=375


,text,primary_emotion,secondary_emotions,meta_emotions,sentiment,interaction_style,intensity,context
801,"Posted my artwork online for the first time, a...",Pride,"['Nervousness', 'Gratitude']","['Reflection on courage', 'Commitment to creat...",Positive,Encouraging,8,Creativity
1074,I’m so proud of my progress in therapy. It has...,Pride,"['Hope', 'Gratitude']","['Reflection on emotional resilience', 'Commit...",Positive,Empowering,8,Mental Health
1458,"I’m happy that I’m learning something new, but...",Excitement,"['Stress', 'Anticipation']","['Reflection on growth', 'Commitment to learni...",Mixed,Conflicted,7,Personal Growth
127,"I was standing in front of the old house, now ...",Sadness,"['Nostalgia', 'Bittersweetness']","['Reflection on past memories', 'Acceptance of...",Mixed,Reflective,8,Memory (Home)
1402,Seeing how far I’ve come since I started this ...,Pride,"['Gratitude', 'Satisfaction']","['Reflection on growth', 'Commitment to self-i...",Positive,Empowering,8,Personal Growth



Available labels: ['Mixed', 'Positive', 'Negative', 'Ambiguous', 'Neutral']


## 1. Data Preparation
Transform our previously processed dataset into a format suitable for transformer models:
1. Load and split data into train/val/test sets
2. Convert data into HuggingFace Dataset format
3. Initialize RoBERTa model and tokenizer
4. Tokenize text data for all splits
5. Set up label mappings for our emotion classes


In [ ]:
def prepare_dataset(df):
    """Convert DataFrame to HuggingFace Dataset format"""
    return Dataset.from_dict({
        'text': df['text'].tolist(),
        'labels': [label_mapping[label] for label in df['sentiment']]
    })

# Convert to HuggingFace datasets
train_dataset = prepare_dataset(train_df)
val_dataset = prepare_dataset(val_df)
test_dataset = prepare_dataset(test_df)

print(f"\nTraining samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

# Load pre-trained model and tokenizer
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_mapping),
    id2label=id2label,
    label2id=label2id
)

# Tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

# Tokenize datasets
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)


Training samples: 1124
Validation samples: 375
Test samples: 375


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 375/375 [00:00<00:00, 17042.81 examples/s]


## 2. Model Training and Evaluation
Configure and execute the training process:
1. Define evaluation metrics (F1-score)
2. Calculate class weights to handle imbalanced data
3. Set up training arguments (learning rate, batch size, etc.)
4. Train the model using HuggingFace Trainer
5. Evaluate model performance on test set
6. Save the final model and tokenizer

In [3]:
## 2. Model Training

# Define metrics
metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    return metric.compute(
        predictions=predictions,
        references=labels,
        average="macro"
    )

# Calculate class weights
total_samples = len(train_dataset)
class_counts = train_dataset.select_columns(['labels']).unique('labels')
class_weights = torch.FloatTensor([
    total_samples / (len(train_dataset.filter(lambda x: x['labels'] == i)) * len(class_counts))
    for i in range(len(label_mapping))
])

# Training arguments
training_args = TrainingArguments(
    output_dir="results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="steps",         # Use eval_strategy instead of evaluation_strategy
    save_strategy="steps",         # Use save_strategy instead of save_strategy
    eval_steps=500,               # How often to evaluate
    save_steps=500,               # How often to save
    logging_dir='./logs',         # Directory for storing logs
    logging_steps=100,            # How often to log
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    push_to_hub=False,
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

# Train the model
trainer.train()

Filter: 100%|██████████| 1124/1124 [00:00<00:00, 224580.68 examples/s]


Step,Training Loss,Validation Loss


TrainOutput(global_step=213, training_loss=0.7179272029321518, metrics={'train_runtime': 88.2088, 'train_samples_per_second': 38.227, 'train_steps_per_second': 2.415, 'total_flos': 221808594097152.0, 'train_loss': 0.7179272029321518, 'epoch': 3.0})

In [4]:
## 3. Model Evaluation

# Get predictions
predictions = trainer.predict(tokenized_test)
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

# Print classification report
print("\nClassification Report:")
print(classification_report(
    labels,
    preds,
    target_names=list(label_mapping.keys()),
    zero_division=0
))

# Save the model
output_dir = project_root / "models" / "transformer_model"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"\nModel saved to {output_dir}")


Classification Report:
              precision    recall  f1-score   support

       Mixed       0.72      0.86      0.78       130
    Positive       0.92      0.97      0.95       108
    Negative       0.85      0.71      0.77        84
   Ambiguous       0.38      0.19      0.25        32
     Neutral       0.83      0.71      0.77        21

    accuracy                           0.79       375
   macro avg       0.74      0.69      0.70       375
weighted avg       0.78      0.79      0.78       375


Model saved to /home/francisco/repositories/Emotion_Classifier/models/transformer_model
